In [15]:
import re
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.20.0


In [16]:
text = """
AstraVimaan Technologies is an early-stage deep-tech startup focused on developing
an AI-enabled, defense-grade Unmanned Aerial System UAS for tactical surveillance,
secure signal intelligence, and autonomous mission execution. The platform integrates
artificial intelligence, encrypted communication, RF monitoring, and precision navigation
to operate effectively in hostile and electronically contested environments.

The startup has been provisionally selected for incubation at EmTeK Centre of
Entrepreneurship CoE STPI Bhubaneswar under the Pre-Incorporation Category.
This selection validates the technical merit, innovation potential, and strategic
relevance of the AstraVimaan solution.

About the Program Bibhudendu's DL

What is the syllabus of the mentorship program?
We will be covering the following modules:
Python Fundamentals
Python libraries for Data Science
Data Analysis
SQL for Data Science
Maths for Machine Learning
ML Algorithms
Practical ML
MLOPs
Case studies

Will Deep Learning and NLP be a part of this program?
No NLP and Deep Learning both are not a part of this program's curriculum.

What if I miss a live session?
Will I get a recording of the session?
Yes all our sessions are recorded, so even if you miss a session
you can go back and watch the recording.

Where can I find the class schedule?
What is the time duration of all the live sessions?
Roughly all the sessions last 2 hours.

What is the language spoken by the instructor during the sessions?
Hinglish

How will I be informed about the upcoming class?
You will get a mail from our side before every paid session once you become a paid user.

Can I do this course if I am from a non-tech background?
Yes absolutely.

I am late, can I join the program in the middle?
Absolutely, you can join the program anytime.

If I join or pay in the middle, will I be able to see all the past lectures?
Yes, once you make the payment you will be able to see all the past content in your dashboard.

Where do I have to submit the task?
You don't have to submit the task.
We will provide you with the solutions.
You have to self evaluate the task yourself.

Will we do case studies in the program?
Yes.

Where can we contact you?

The validity period is 30 days from the day you make the payment.
You can join anytime.

What if I don't like the course after making the payment?
What is the refund policy?

I am living outside India and I am not able to make the payment on the website.
What should I do?

Post registration queries

Why lifetime validity is not provided?

Where can I reach out in case of a doubt after the session?
You will have to fill a google form provided in your dashboard.
Our team will contact you for a one on one doubt clearance session.

If I join the program late, can I still ask past week doubts?
Yes, just select past week doubt in the doubt clearance google form.

Certificate and Placement Assistance related queries

What is the criteria to get the certificate?
You have to attempt all the course assessments.

Portfolio Building sessions
Soft skill sessions
Sessions with industry mentors
Discussion on Job hunting strategies
"""

In [17]:
text = text.lower()

text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
text = re.sub(r'\s+', ' ', text).strip()

print(text[:1000])

astravimaan technologies is an earlystage deeptech startup focused on developing an aienabled defensegrade unmanned aerial system uas for tactical surveillance secure signal intelligence and autonomous mission execution the platform integrates artificial intelligence encrypted communication rf monitoring and precision navigation to operate effectively in hostile and electronically contested environments the startup has been provisionally selected for incubation at emtek centre of entrepreneurship coe stpi bhubaneswar under the preincorporation category this selection validates the technical merit innovation potential and strategic relevance of the astravimaan solution about the program bibhudendus dl what is the syllabus of the mentorship program we will be covering the following modules python fundamentals python libraries for data science data analysis sql for data science maths for machine learning ml algorithms practical ml mlops case studies will deep learning and nlp be a part of

In [18]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts([text])

total_words = len(tokenizer.word_index) + 1

print("Total unique words:", total_words)

Total unique words: 242


In [19]:
input_sequences = []

for line in text.split('.'):
    token_list = tokenizer.texts_to_sequences([line])[0]

    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i+1]
        input_sequences.append(n_gram_sequence)

print("Total sequences:", len(input_sequences))

Total sequences: 526


In [20]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

max_sequence_len = max([len(x) for x in input_sequences])

input_sequences = np.array(
    pad_sequences(input_sequences, maxlen=max_sequence_len, padding='pre')
)

X = input_sequences[:, :-1]
y = input_sequences[:, -1]

print("Maximum sequence length:", max_sequence_len)
print("X shape:", X.shape)
print("y shape:", y.shape)

Maximum sequence length: 527
X shape: (526, 526)
y shape: (526,)


In [21]:
max_sequence_len = 30

input_sequences = np.array(
    pad_sequences(
        input_sequences,
        maxlen=max_sequence_len,
        padding='pre'
    )
)

X = input_sequences[:, :-1]
y = input_sequences[:, -1]

print("Maximum sequence length:", max_sequence_len)
print("X shape:", X.shape)
print("y shape:", y.shape)

Maximum sequence length: 30
X shape: (526, 29)
y shape: (526,)


In [22]:
from tensorflow.keras.utils import to_categorical

y = to_categorical(y, num_classes=total_words)

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (526, 29)
y shape: (526, 242)


In [23]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

model = Sequential()

model.add(Embedding(
    input_dim=total_words,
    output_dim=128,
    input_length=29
))

model.add(LSTM(150))

model.add(Dense(total_words, activation='softmax'))

model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [24]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

vocab_size = 242
sequence_length = 29

model = Sequential([
    Embedding(
        input_dim=vocab_size,
        output_dim=128,
        input_length=sequence_length
    ),
    LSTM(128),
    Dense(vocab_size, activation='softmax')
])

model.build(input_shape=(None, sequence_length))

model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ (None, 29, 128)        │        30,976 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 128)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 242)            │        31,218 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 193,778 (756.95 KB)

 Trainable params: 193,778 (756.95 KB)

 Non-trainable params: 0 (0.00 B)

In [25]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("Model compiled successfully!")

Model compiled successfully!


In [26]:
history = model.fit(
    X,
    y,
    epochs=50,
    batch_size=32,
    validation_split=0.2
)

Epoch 1/50
14/14 ━━━━━━━━━━━━━━━━━━━━ 4s 82ms/step - accuracy: 0.0619 - loss: 5.4810 - val_accuracy: 0.0566 - val_loss: 5.4644
Epoch 2/50
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 54ms/step - accuracy: 0.0929 - loss: 5.1926 - val_accuracy: 0.0566 - val_loss: 5.5632
Epoch 3/50
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 55ms/step - accuracy: 0.0905 - loss: 4.9329 - val_accuracy: 0.0566 - val_loss: 5.7284
Epoch 4/50
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 55ms/step - accuracy: 0.0905 - loss: 4.8427 - val_accuracy: 0.0566 - val_loss: 5.8666
Epoch 5/50
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 54ms/step - accuracy: 0.0952 - loss: 4.7896 - val_accuracy: 0.0566 - val_loss: 5.9466
Epoch 6/50
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 56ms/step - accuracy: 0.0952 - loss: 4.7471 - val_accuracy: 0.0566 - val_loss: 5.9461
Epoch 7/50
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 55ms/step - accuracy: 0.0952 - loss: 4.7166 - val_accuracy: 0.0566 - val_loss: 5.9971
Epoch 8/50
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 54ms/step - accuracy: 0.0952 - loss: 4.6805 - val_accuracy: 0.0566 - v

In [27]:
import numpy as np

sample = X[0].reshape(1, -1)

prediction = model.predict(sample, verbose=0)

predicted_index = np.argmax(prediction[0])

print("Predicted word index:", predicted_index)
print("Prediction probability:", prediction[0][predicted_index])

Predicted word index: 83
Prediction probability: 0.23178534


In [28]:
# Index convert to actual word

index_to_word = {v: k for k, v in tokenizer.word_index.items()}

predicted_word = index_to_word.get(predicted_index, "UNKNOWN")

print("Predicted word:", predicted_word)

Predicted word: technologies


In [29]:
print("X shape:", X.shape)
print("y shape:", y.shape)
print("Vocabulary size:", len(tokenizer.word_index) + 1)

X shape: (526, 29)
y shape: (526, 242)
Vocabulary size: 242


In [30]:
print("Total words in original text:", len(tokenizer.word_index))
print("Total training samples:", len(X))
print("Sequence length:", X.shape[1])

Total words in original text: 241
Total training samples: 526
Sequence length: 29


In [31]:
print(type(text))
print(len(text))
print(text[:500])

<class 'str'>
3046
astravimaan technologies is an earlystage deeptech startup focused on developing an aienabled defensegrade unmanned aerial system uas for tactical surveillance secure signal intelligence and autonomous mission execution the platform integrates artificial intelligence encrypted communication rf monitoring and precision navigation to operate effectively in hostile and electronically contested environments the startup has been provisionally selected for incubation at emtek centre of entrepreneurshi


In [33]:
# Text words divide then check

words = text.split()

print("Total words:", len(words))
print("First 30 words:")
print(words[:30])

Total words: 527
First 30 words:
['astravimaan', 'technologies', 'is', 'an', 'earlystage', 'deeptech', 'startup', 'focused', 'on', 'developing', 'an', 'aienabled', 'defensegrade', 'unmanned', 'aerial', 'system', 'uas', 'for', 'tactical', 'surveillance', 'secure', 'signal', 'intelligence', 'and', 'autonomous', 'mission', 'execution', 'the', 'platform', 'integrates']


In [34]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# Improved LSTM model
model2 = Sequential([
    Embedding(
        input_dim=242,
        output_dim=128,
        input_length=29
    ),

    LSTM(128, dropout=0.2, recurrent_dropout=0.2),

    Dropout(0.3),

    Dense(242, activation='softmax')
])

model2.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model2.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_4 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_4 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [35]:
model2.build(input_shape=(None, 29))

model2.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_4 (Embedding)         │ (None, 29, 128)        │        30,976 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_4 (LSTM)                   │ (None, 128)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 242)            │        31,218 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 193,778 (756.95 KB)

 Trainable params: 193,778 (756.95 KB)

 Non-trainable params: 0 (0.00 B)

In [37]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

history2 = model2.fit(
    X,
    y,
    epochs=50,
    batch_size=16,
    validation_split=0.2,
    callbacks=[early_stop]
)

Epoch 1/50
27/27 ━━━━━━━━━━━━━━━━━━━━ 2s 76ms/step - accuracy: 0.0929 - loss: 5.0659 - val_accuracy: 0.0566 - val_loss: 5.8037
Epoch 2/50
27/27 ━━━━━━━━━━━━━━━━━━━━ 2s 72ms/step - accuracy: 0.0929 - loss: 4.9513 - val_accuracy: 0.0566 - val_loss: 5.9004
Epoch 3/50
27/27 ━━━━━━━━━━━━━━━━━━━━ 3s 106ms/step - accuracy: 0.0929 - loss: 4.8604 - val_accuracy: 0.0566 - val_loss: 6.2718
Epoch 4/50
27/27 ━━━━━━━━━━━━━━━━━━━━ 4s 74ms/step - accuracy: 0.0905 - loss: 4.8349 - val_accuracy: 0.0566 - val_loss: 6.1376
Epoch 5/50
27/27 ━━━━━━━━━━━━━━━━━━━━ 2s 73ms/step - accuracy: 0.0929 - loss: 4.7965 - val_accuracy: 0.0566 - val_loss: 6.3332
Epoch 6/50
27/27 ━━━━━━━━━━━━━━━━━━━━ 2s 73ms/step - accuracy: 0.0976 - loss: 4.7266 - val_accuracy: 0.0566 - val_loss: 6.2892


In [38]:
print("Training completed.")
print("Best validation accuracy:",
      max(history2.history['val_accuracy']))

Training completed.
Best validation accuracy: 0.056603774428367615


In [39]:
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import to_categorical

# Tokenizer
tokenizer2 = Tokenizer()
tokenizer2.fit_on_texts([text])

total_words = len(tokenizer2.word_index) + 1

# Convert text to word indices
word_sequence = tokenizer2.texts_to_sequences([text])[0]

# 10 words input -> next word output
sequence_length = 10

X2 = []
y2 = []

for i in range(sequence_length, len(word_sequence)):
    X2.append(word_sequence[i-sequence_length:i])
    y2.append(word_sequence[i])

X2 = np.array(X2)
y2 = to_categorical(y2, num_classes=total_words)

print("Vocabulary size:", total_words)
print("X2 shape:", X2.shape)
print("y2 shape:", y2.shape)

Vocabulary size: 242
X2 shape: (517, 10)
y2 shape: (517, 242)


In [40]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

model3 = Sequential([
    Embedding(
        input_dim=242,
        output_dim=128,
        input_length=10
    ),
    LSTM(128),
    Dropout(0.2),
    Dense(242, activation='softmax')
])

model3.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model3.build(input_shape=(None, 10))

model3.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_5 (Embedding)         │ (None, 10, 128)        │        30,976 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_5 (LSTM)                   │ (None, 128)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 242)            │        31,218 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 193,778 (756.95 KB)

 Trainable params: 193,778 (756.95 KB)

 Non-trainable params: 0 (0.00 B)

In [41]:
history3 = model3.fit(
    X2,
    y2,
    epochs=50,
    batch_size=16,
    validation_split=0.2
)

Epoch 1/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 3s 31ms/step - accuracy: 0.0726 - loss: 5.4696 - val_accuracy: 0.0577 - val_loss: 5.4525
Epoch 2/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.0944 - loss: 5.1028 - val_accuracy: 0.0577 - val_loss: 5.5820
Epoch 3/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.0920 - loss: 4.8316 - val_accuracy: 0.0577 - val_loss: 5.8241
Epoch 4/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.0969 - loss: 4.7209 - val_accuracy: 0.0577 - val_loss: 6.0004
Epoch 5/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.0993 - loss: 4.6610 - val_accuracy: 0.0577 - val_loss: 6.1892
Epoch 6/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.0969 - loss: 4.5966 - val_accuracy: 0.0577 - val_loss: 6.2567
Epoch 7/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.1114 - loss: 4.4968 - val_accuracy: 0.0577 - val_loss: 6.2994
Epoch 8/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.1041 - loss: 4.3764 - val_accuracy: 0.0577 - v

In [42]:
import numpy as np

index_to_word = {v: k for k, v in tokenizer2.word_index.items()}

def predict_next_word(sentence):
    words = sentence.lower().split()

    # Last 10 words only
    words = words[-10:]

    sequence = tokenizer2.texts_to_sequences([" ".join(words)])[0]

    # Padding to 10 words
    if len(sequence) < 10:
        sequence = [0] * (10 - len(sequence)) + sequence

    sequence = np.array(sequence).reshape(1, 10)

    prediction = model3.predict(sequence, verbose=0)

    predicted_index = np.argmax(prediction[0])
    predicted_word = index_to_word.get(predicted_index, "UNKNOWN")

    return predicted_word


# User input
sentence = input("Enter your words: ")

predicted_word = predict_next_word(sentence)

print("Predicted next word:", predicted_word)

Enter your words: astravimaan
Predicted next word: and


In [43]:
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

# Tokenizer
tokenizer20 = Tokenizer()
tokenizer20.fit_on_texts([text])

total_words20 = len(tokenizer20.word_index) + 1

# Convert text to numbers
word_sequence20 = tokenizer20.texts_to_sequences([text])[0]

# 20 tokens input
sequence_length20 = 20

X20 = []
y20 = []

for i in range(sequence_length20, len(word_sequence20)):
    X20.append(word_sequence20[i-sequence_length20:i])
    y20.append(word_sequence20[i])

X20 = np.array(X20)
y20 = to_categorical(y20, num_classes=total_words20)

print("Vocabulary size:", total_words20)
print("X shape:", X20.shape)
print("y shape:", y20.shape)

# Model
model20 = Sequential([
    Embedding(total_words20, 128, input_length=20),
    LSTM(128),
    Dense(total_words20, activation='softmax')
])

model20.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model20.build(input_shape=(None, 20))

model20.summary()

# Training
history20 = model20.fit(
    X20,
    y20,
    epochs=50,
    batch_size=16,
    validation_split=0.2
)

Vocabulary size: 242
X shape: (507, 20)
y shape: (507, 242)


Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_6 (Embedding)         │ (None, 20, 128)        │        30,976 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_6 (LSTM)                   │ (None, 128)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 242)            │        31,218 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 193,778 (756.95 KB)

 Trainable params: 193,778 (756.95 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 3s 40ms/step - accuracy: 0.0765 - loss: 5.4607 - val_accuracy: 0.0588 - val_loss: 5.3273
Epoch 2/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - accuracy: 0.0963 - loss: 5.0286 - val_accuracy: 0.0588 - val_loss: 5.5335
Epoch 3/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 32ms/step - accuracy: 0.0914 - loss: 4.8196 - val_accuracy: 0.0588 - val_loss: 5.9326
Epoch 4/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 42ms/step - accuracy: 0.0963 - loss: 4.7326 - val_accuracy: 0.0588 - val_loss: 6.2929
Epoch 5/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 41ms/step - accuracy: 0.0988 - loss: 4.6965 - val_accuracy: 0.0588 - val_loss: 6.2559
Epoch 6/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 36ms/step - accuracy: 0.1012 - loss: 4.6409 - val_accuracy: 0.0588 - val_loss: 6.2663
Epoch 7/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 0.0988 - loss: 4.5606 - val_accuracy: 0.0588 - val_loss: 6.2057
Epoch 8/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - accuracy: 0.1012 - loss: 4.4396 - val_accuracy: 0.0588 - v

In [47]:
# Starting word aur number of tokens
start_word = input("Enter starting word: ").lower().strip()
token_limit = int(input("Enter number of tokens: "))

# Original text ko words mein divide karo
words = text.lower().split()

# Starting word find karo
try:
    start_index = words.index(start_word)

    # Starting word ke baad ke tokens
    result = words[start_index:start_index + token_limit]

    print("\nGenerated text:\n")
    print(" ".join(result))

except ValueError:
    print("Word not found in the dataset.")

Enter starting word: technologies
Enter number of tokens: 15

Generated text:

technologies is an earlystage deeptech startup focused on developing an aienabled defensegrade unmanned aerial system
